<a href="https://colab.research.google.com/github/sahara-healthcare-suite/sahara-healthcare-suite/blob/main/AfriHealth_Sahara_Benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install requests pandas jiwer transformers accelerate datasets soundfile librosa python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 63.7 MB/s eta 0:00:00


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled")

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [ ]:
import os

folders = [
    "data",
    "data/audio",
    "results",
    "models"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created.")

Folders created.


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving CS-02'.m4a to CS-02'.m4a
Saving CS-02.m4a to CS-02.m4a
Saving CS-01.m4a to CS-01.m4a
Saving CS_04.m4a to CS_04.m4a
Saving CS_03.m4a to CS_03.m4a


In [ ]:
import shutil
import os

filename = list(uploaded.keys())[0]

shutil.move(
    filename,
    f"data/audio/{filename}"
)

print("Saved:", f"data/audio/{filename}")

Saved: data/audio/CS-02'.m4a


In [ ]:
from IPython.display import Audio, display

audio_path = f"data/audio/{filename}"

display(Audio(audio_path))

In [ ]:
reference_transcript = """
The child has high fever for three days እና በጣም ደክሟል.
""".strip()

print(reference_transcript)

The child has high fever for three days እና በጣም ደክሟል.


In [ ]:
!pip install python-dotenv

import os
from dotenv import load_dotenv

load_dotenv()  # Loads variables from .env file into os.environ

api_key = os.environ.get("INTRON_API_KEY")
print("API key loaded:", api_key is not None)

API key loaded: False


In [ ]:
import requests
import os

def transcribe_sahara(audio_path):

    url = "https://infer.voice.intron.io/file/v1/upload/sync"

    headers = {
        "Authorization": f"Bearer {os.environ['INTRON_API_KEY']}"
    }

    with open(audio_path, "rb") as audio:

        files = {
            "audio_file_blob": audio
        }

        data = {
            "audio_file_name": os.path.basename(audio_path),
            "use_language_asr_input": "am",
            "use_category": "file_category_telehealth"
        }

        response = requests.post(
            url,
            headers=headers,
            files=files,
            data=data,
            timeout=120
        )

    print("HTTP status:", response.status_code)

    response.raise_for_status()

    result = response.json()

    return result["data"]["audio_transcript"]

In [ ]:
os.environ['INTRON_API_KEY'] = "YOUR_ACTUAL_INTRON_API_KEY"

In [ ]:
import os

# Put your real API key string inside the quotes in cell [20] or a single cell:
os.environ['INTRON_API_KEY'] = "intr_a9f6d5f21e7aae89f1b5bef5a2a0954b1c1027cb33328015"

sahara_transcript = transcribe_sahara(audio_path)
print("SAHARA TRANSCRIPT")
print("================")
print(sahara_transcript)

HTTP status: 200
SAHARA TRANSCRIPT
Prescribe Amoxicline 500mg Capsules TID for 7 days ከምገባዋል



In [ ]:
# 1. Install the missing package
!pip install -q jiwer

# 2. Import functions
from jiwer import wer, cer

# 3. Calculate WER and CER
sahara_wer = wer(
    reference_transcript,
    sahara_transcript
)

sahara_cer = cer(
    reference_transcript,
    sahara_transcript
)

# 4. Print results (formatted properly using f-strings)
print(f"Sahara WER: {round(sahara_wer * 100, 2)}%")
print(f"Sahara CER: {round(sahara_cer * 100, 2)}%")

Sahara WER: 81.82%
Sahara CER: 86.54%


In [ ]:
from jiwer import wer, cer

sahara_wer = wer(
    reference_transcript,
    sahara_transcript
)

sahara_cer = cer(
    reference_transcript,
    sahara_transcript
)

print("Sahara WER:", round(sahara_wer * 100, 2), "%")
print("Sahara CER:", round(sahara_cer * 100, 2), "%")

Sahara WER: 81.82 %
Sahara CER: 86.54 %


In [ ]:
from transformers import pipeline

whisper = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium",
    device=0 if torch.cuda.is_available() else -1
)

config.json:   0%|          | 0.00/1.99k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.06GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.75k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

In [ ]:
!apt-get install -y ffmpeg
!pip install -q librosa soundfile

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.


In [ ]:
import os
import torch
import torchaudio

# 1. Verify file exists
if not os.path.exists(audio_path):
    raise FileNotFoundError(f"File not found: {audio_path}")

# 2. Load audio using torchaudio (handles formatting and resamples safely)
waveform, sample_rate = torchaudio.load(audio_path)

# Convert stereo to mono if needed
if waveform.shape[0] > 1:
    waveform = torch.mean(waveform, dim=0, keepdim=True)

# Resample to 16kHz for Whisper
if sample_rate != 16000:
    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
    waveform = resampler(waveform)

clean_audio_path = "converted_audio.wav"
torchaudio.save(clean_audio_path, waveform, 16000)

# 3. Transcribe using the converted audio
whisper_result = whisper(clean_audio_path)
whisper_transcript = whisper_result["text"]

print("WHISPER TRANSCRIPT")
print("==================")
print(whisper_transcript)

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

WHISPER TRANSCRIPT
 Prescribe Amoxa Sling 500mg Capsules TID for 7 days. Come get boiled.


In [ ]:
whisper_wer = wer(
    reference_transcript,
    whisper_transcript
)

whisper_cer = cer(
    reference_transcript,
    whisper_transcript
)

print("Whisper WER:", round(whisper_wer * 100, 2), "%")
print("Whisper CER:", round(whisper_cer * 100, 2), "%")

Whisper WER: 100.0 %
Whisper CER: 92.31 %


In [ ]:
import pandas as pd

results = pd.DataFrame([
    {
        "model": "Intron Sahara v2.5",
        "WER": sahara_wer,
        "CER": sahara_cer
    },
    {
        "model": "OpenAI Whisper Medium",
        "WER": whisper_wer,
        "CER": whisper_cer
    }
])

results

,model,WER,CER
0,Intron Sahara v2.5,0.818182,0.865385
1,OpenAI Whisper Medium,1.000000,0.923077


In [ ]:
results["WER_percent"] = results["WER"] * 100
results["CER_percent"] = results["CER"] * 100

results

,model,WER,CER,WER_percent,CER_percent
0,Intron Sahara v2.5,0.818182,0.865385,81.818182,86.538462
1,OpenAI Whisper Medium,1.000000,0.923077,100.000000,92.307692


In [ ]:
def triage_classifier(text):

    text = text.lower()

    emergency_terms = [
        "severe difficulty breathing",
        "unconscious",
        "severe bleeding",
        "seizure",
        "cannot breathe"
    ]

    urgent_terms = [
        "high fever",
        "chest pain",
        "dehydration",
        "persistent vomiting",
        "difficulty breathing"
    ]

    for term in emergency_terms:
        if term in text:
            return "EMERGENCY"

    for term in urgent_terms:
        if term in text:
            return "URGENT"

    return "ROUTINE"

In [ ]:
print(
    "Sahara triage:",
    triage_classifier(sahara_transcript)
)

print(
    "Whisper triage:",
    triage_classifier(whisper_transcript)
)

Sahara triage: ROUTINE
Whisper triage: ROUTINE
